# E179 full instructor demonstration and reference solution
 # Copy or Move this notebook to the root of the repo before running
Private instructor material. Run all cells top to bottom in E179 Assessment (CPU). This notebook executes every technical activity and provides interpretation for all three tasks. Outputs are reference evidence, not a single mandatory student answer. It uses no LLM, Bedrock service, container, endpoint, or GPU.

The source is an older public phishing-feature dataset. All stress and shift fixtures are explicitly simulated. Expected numerical results are recorded in the executed copy and REFERENCE_FINDINGS.md. Minor runtime/latency differences are normal. Default execution also rebuilds models into a private directory to verify reproducibility without changing the student artifacts.

In [ ]:
from pathlib import Path
import sys, os, json
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['NUMBA_NUM_THREADS']='2'
os.environ['DO_NOT_TRACK']='1'
# The notebooks are opened at the repository root; assets are under support/.
cwd=Path.cwd().resolve()
ROOT=next(p for p in [cwd/'support', cwd.parent/'support', cwd.parent/'repository'/'support', cwd] if (p/'labkit.py').exists())
sys.path.insert(0,str(ROOT))
import numpy as np, pandas as pd
from labkit import FEATURES, load, load_models, recipes, metrics, cross_validate, wilson
models=load_models()
print('Loaded pretrained models:',list(models))
print('Python:',sys.version.split()[0], 'features:',len(FEATURES))

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
OUT=cwd/'reference_outputs'; OUT.mkdir(exist_ok=True)
def save(frame,name):
    frame.to_csv(OUT/name,index=False)
    display(frame)
training=load('training'); validation=load('validation')
manifest=json.loads((ROOT/'data/manifest.json').read_text())
display(manifest)

## Task 1 Requirements and provisional selection
Example role: AI/ML Security Assurance Engineer, assessing phishing triage before SOC review. Scenario assumptions: 1.2 million already-extracted feature vectors/day, 0.1% phishing prevalence, warm scoring P95 within 250 ms and capacity for 15,000 daily alerts. A proposed target is at least 90% recall at no more than 1% FPR; these are fictional business targets, not dataset facts. Total pipeline latency includes feature acquisition, which cannot be measured with these features alone.

An acceptable provisional choice is Logistic Regression for simplicity, followed by evidence-based reassessment against XGBoost. XGBoost can represent nonlinear interactions; extra capacity may improve fit but does not guarantee generalisation. No complexity-sweep retraining is assigned. Select a model only subject to operational evidence, contemporary representative validation and governance approval. The old dataset cannot justify immediate production blocking.

## Verify pretrained model provenance and loading
The Logistic artifact stores the fitted one-hot encoder and coefficients together. The XGBoost UBJSON stores trees and objective; schema, class mapping and recipe are in provenance.json and labkit.py. Only the training split was used for fitting. Load trusted joblib files only: deserialisation can execute code. The version and digest checks avoid accidental environment/artifact mismatch but are not a substitute for a trusted distribution channel.

In [ ]:
provenance=json.loads((ROOT/'models/provenance.json').read_text())
display(provenance)
assert provenance['training_rows']==len(training)
assert len(FEATURES)==30
assert set(training.label)=={0,1}

## Publisher training and export demonstration
This is instructor preparation, not a student training exercise. Fit the frozen recipes using training only, export, reload and compare scores. No training happens in ordinary student model loading. The package contains the fitted files already, so future runs can set REBUILD_MODELS=False.

In [ ]:
REBUILD_MODELS=True
if REBUILD_MODELS:
    import joblib, importlib.metadata as md
    from labkit import sha
    rebuilt=OUT/'rebuilt_models'; rebuilt.mkdir(exist_ok=True)
    fitted=recipes()
    for m in fitted.values(): m.fit(training[FEATURES],training.label)
    joblib.dump(fitted['Logistic'],rebuilt/'logistic.joblib',compress=3)
    fitted['XGBoost'].save_model(rebuilt/'xgboost.ubj')
    meta=dict(provenance)
    meta['artifact_sha256']={f:sha(rebuilt/f) for f in ['logistic.joblib','xgboost.ubj']}
    (rebuilt/'provenance.json').write_text(json.dumps(meta,indent=2))
    reloaded=load_models(rebuilt)
    for name in models:
        np.testing.assert_allclose(models[name].predict_proba(validation[FEATURES]),reloaded[name].predict_proba(validation[FEATURES]),atol=1e-7)
    print('Training, export, reload and prediction parity passed')

## Task 2 Evaluation protocol and cross validation
Keep final holdout untouched until threshold selection is complete. Clean splits have no identical feature vectors across partitions. This is still not a domain-separated or chronological test: source domains and timestamps are unavailable. Five-fold stratified cross-validation refits fresh pipelines using only training, including preprocessing in each fold. This estimates the fixed learning procedure, whereas holdout evaluates supplied fitted artifacts. Fold variation is descriptive, not a confidence interval from independent experiments.

In [ ]:
cv=cross_validate(training)
save(cv,'cv_folds.csv')
display(cv.groupby('model')[['recall','fpr','f1','average_precision']].agg(['mean','std']))

## Select thresholds on validation and record the decision
Illustrative rule: choose the threshold with highest validation recall among a predeclared grid satisfying FPR <= 0.01. If none qualify, use 1.0 and explicitly flag infeasibility. The 90% recall target must still be checked. Another justified, predeclared rule is acceptable. We do not tune using final holdout, vendor holdout scores or stress outcomes.

In [ ]:
thresholds={}; selections=[]
for name,model in models.items():
    p=model.predict_proba(validation[FEATURES])[:,1]
    grid=pd.DataFrame([metrics(validation.label,p,t) for t in np.linspace(.05,.99,95)])
    feasible=grid[grid.fpr<=.01]
    row=feasible.sort_values(['recall','threshold'],ascending=[False,False]).iloc[0] if len(feasible) else pd.Series(metrics(validation.label,p,1.0))
    thresholds[name]=float(row.threshold)
    selections.append(dict(model=name,feasible=bool(len(feasible)),**row.to_dict()))
    grid.to_csv(OUT/f'{name}_validation_thresholds.csv',index=False)
save(pd.DataFrame(selections),'validation_selection.csv')
(OUT/'decision_lock.json').write_text(json.dumps({'thresholds':thresholds,'protocol':'validation FPR <= .01; maximise recall; fixed grid; seed 179'},indent=2))
# Now release final holdout.
holdout=load('holdout')
for other in [training,validation]:
    assert not set(holdout.event_id)&set(other.event_id)
    assert not set(map(tuple,holdout[FEATURES].values))&set(map(tuple,other[FEATURES].values))

## Final performance and uncertainty
Phishing is positive. Report counts as well as rates. Average precision summarises the precision-recall curve; it is not trapezoidal PR-AUC. Wilson intervals are approximate binomial intervals, conditional on this curated sample and its independence assumptions. Zero observed false positives does not prove a zero population FPR.

In [ ]:
result=[]; probabilities={}
for name,m in models.items():
    probabilities[name]=m.predict_proba(holdout[FEATURES])[:,1]
    result.append(dict(model=name,**metrics(holdout.label,probabilities[name],thresholds[name])))
results=pd.DataFrame(result); save(results,'holdout_metrics.csv')
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,(name,p) in zip(axes,probabilities.items()):
    ConfusionMatrixDisplay.from_predictions(holdout.label,p>=thresholds[name],ax=ax,colorbar=False)
    ax.set_title(name)
fig.tight_layout();fig.savefig(OUT/'confusion.png',dpi=130);plt.show()
fig,ax=plt.subplots(figsize=(6,4))
for name,p in probabilities.items():PrecisionRecallDisplay.from_predictions(holdout.label,p,name=name,ax=ax)
fig.tight_layout();fig.savefig(OUT/'precision_recall.png',dpi=130);plt.show()

## Operational subgroup analysis
Use source feature strata (SSLfinal_State and web_traffic) rather than inventing demographic labels. Compare false-positive and false-negative rates with class denominators and uncertainty. These features are also predictors; conditional differences may partly reflect model dependence and sample composition. This is operational bias analysis, not proof of demographic fairness or causal discrimination.

In [ ]:
groups=[]
for name,m in models.items():
    for col in ['SSLfinal_State','web_traffic']:
        for group,d in holdout.groupby(col):
            groups.append(dict(model=name,stratum=col,group=int(group),**metrics(d.label,m.predict_proba(d[FEATURES])[:,1],thresholds[name])))
save(pd.DataFrame(groups),'subgroups.csv')

## Explainability with SHAP
TreeExplainer attributes the XGBoost raw output (log odds) relative to its background expectation. Use a bounded sample so CPU cost stays low. Global importance is mean absolute attribution over that sample. A local waterfall explains one prediction, not its correctness. Feature correlations and old heuristics limit causal interpretation. Expect TLS/anchor-related features to be influential; inspect the actual result rather than assuming it.

In [ ]:
import shap
x=holdout[FEATURES].iloc[:120]
explainer=shap.TreeExplainer(models['XGBoost'])
sv=explainer(x)
importance=pd.DataFrame({'feature':FEATURES,'mean_abs_shap':np.abs(sv.values).mean(axis=0)}).sort_values('mean_abs_shap',ascending=False)
save(importance,'shap_importance.csv')
shap.plots.bar(sv,show=False,max_display=12);plt.tight_layout();plt.savefig(OUT/'shap_global.png',dpi=130,bbox_inches='tight');plt.show()
shap.plots.waterfall(sv[0],show=False,max_display=10);plt.savefig(OUT/'shap_local.png',dpi=130,bbox_inches='tight');plt.show()

## LIME local approximation and repeatability
These are categorical heuristic codes, not continuous quantities. Recode [-1,0,1] to [0,1,2] for LIME and map back before prediction. Declare all features categorical. Compare two random seeds and report local surrogate fit. Perturbed combinations can be unrealistic; disagreement or a weak fit reduces confidence in the explanation.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
lime_rows=[]
def lime_predict(z):
    frame=pd.DataFrame(np.rint(z).astype(int)-1,columns=FEATURES)
    return models['XGBoost'].predict_proba(frame)
for seed in [179,271]:
    le=LimeTabularExplainer(training[FEATURES].to_numpy()+1,feature_names=FEATURES,class_names=['legitimate','phishing'],categorical_features=list(range(len(FEATURES))),categorical_names={i:['code -1','code 0','code 1'] for i in range(len(FEATURES))},discretize_continuous=False,random_state=seed,mode='classification')
    explanation=le.explain_instance(holdout[FEATURES].iloc[0].to_numpy()+1,lime_predict,labels=(1,),num_features=8,num_samples=1200)
    explanation.save_to_file(str(OUT/f'lime_{seed}.html'))
    lime_rows.append({'seed':seed,'surrogate_r2':explanation.score,'terms':str(explanation.as_list(label=1))})
save(pd.DataFrame(lime_rows),'lime_stability.csv')

## Robustness under a supplied feature-space stress scenario
Pair each malicious row with its baseline by event_id. The synthetic transformation replaces five indicators with legitimate-looking codes. It assumes malicious intent remains unchanged. This is a sensitivity/evasion stress fixture, not verified attack execution or a guarantee that each feature combination is realisable. Since all cases are positive, accuracy equals recall here and FPR/AUC are undefined. Report both recall change and the proportion of initially detected attacks that evade after modification.

In [ ]:
baseline=load('attack_baseline').set_index('event_id').sort_index()
attack=load('attack_data').set_index('event_id').sort_index()
assert baseline.index.equals(attack.index) and baseline.label.equals(attack.label)
robust=[]
for name,m in models.items():
    before=m.predict_proba(baseline[FEATURES])[:,1]>=thresholds[name]
    after=m.predict_proba(attack[FEATURES])[:,1]>=thresholds[name]
    robust.append({'model':name,'n':len(before),'baseline_recall':before.mean(),'stress_recall':after.mean(),'recall_change':after.mean()-before.mean(),'detected_before':int(before.sum()),'evaded':int((before&~after).sum()),'conditional_evasion_rate':float((before&~after).sum()/before.sum()) if before.sum() else np.nan})
save(pd.DataFrame(robust),'robustness.csv')

## Simulated distribution shift
shifted_data.csv resamples holdout with extra weight for SSLfinal_State=1. It is a controlled composition change with repeated rows, not data gathered 90 days later. Total variation distance compares categorical frequency distributions. Evidently supplies an additional exploratory drift report. Drift does not itself prove loss of accuracy. Measure performance on labelled fixtures but avoid treating them as fresh independent evidence or making temporal claims.

In [ ]:
shifted=load('shifted_data'); drift=[]
for col in FEATURES:
    a=training[col].value_counts(normalize=True).reindex([-1,0,1],fill_value=0)
    b=shifted[col].value_counts(normalize=True).reindex([-1,0,1],fill_value=0)
    drift.append({'feature':col,'total_variation':float(abs(a-b).sum()/2)})
save(pd.DataFrame(drift).sort_values('total_variation',ascending=False),'drift_effect_sizes.csv')
from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset
schema=DataDefinition(categorical_columns=FEATURES)
report=Report([DataDriftPreset()])
snapshot=report.run(reference_data=Dataset.from_pandas(training[FEATURES],data_definition=schema),current_data=Dataset.from_pandas(shifted[FEATURES],data_definition=schema))
snapshot.save_html(str(OUT/'drift.html'))
save(pd.DataFrame([dict(model=name,**metrics(shifted.label,m.predict_proba(shifted[FEATURES])[:,1],thresholds[name])) for name,m in models.items()]),'shift_metrics.csv')
print('Shift draws',len(shifted),'unique source events',shifted.event_id.nunique())

## Workload and local timing
Project workload at the scenario prevalence, not the enriched benchmark prevalence. These estimates assume recall and FPR transfer unchanged; test sensitivity to that assumption. Measure warm, single-row predict_proba latency with already-loaded models and feature vectors. It excludes feature acquisition, queues, network, concurrency and service uptime. This cannot certify the production 250 ms SLA. Cost remains an estimate to obtain, not an invented AWS quote.

In [ ]:
import time
work=[];timing=[]
for r in results.to_dict('records'):
    for prevalence in [.0005,.001,.005]:
        tp=1200000*prevalence*r['recall']; fp=1200000*(1-prevalence)*r['fpr']
        work.append({'model':r['model'],'prevalence':prevalence,'true_alerts_day':tp,'false_alerts_day':fp,'missed_phishing_day':1200000*prevalence*(1-r['recall']),'alerts_day':tp+fp,'projected_precision':tp/(tp+fp) if tp+fp else np.nan})
for name,m in models.items():
    one=holdout[FEATURES].iloc[:1]
    for _ in range(10):m.predict_proba(one)
    times=[]
    for _ in range(150):
        start=time.perf_counter();m.predict_proba(one);times.append((time.perf_counter()-start)*1000)
    timing.append({'model':name,'warm_local_p95_ms':np.percentile(times,95),'samples':len(times),'production_sla_verified':False})
save(pd.DataFrame(work),'workload.csv');save(pd.DataFrame(timing),'local_timing.csv')

## Task 3 Audit vendor evidence under black-box constraints
The supplier packet is fictional. Repeated measurements do not create new independent test cases. Reconstruct the headline score, count unique events, check overlap with development data and recompute uncertainty on unique cases. Even the corrected interval does not remove easy-case selection bias or training contamination.

The independent scores permit matched comparison without weights or arbitrary new queries. Compare at disclosed thresholds, explain threshold differences, and ask for further evidence rather than assuming the implementation is safe.

In [ ]:
vendor=pd.read_csv(ROOT/'vendor/benchmark_rows.csv')
unique=vendor.drop_duplicates('event_id')
audit=pd.DataFrame([{'reported_rows':len(vendor),'unique_events':len(unique),'headline_accuracy':float((vendor.label==vendor.prediction).mean()),'unique_accuracy':float((unique.label==unique.prediction).mean()),'training_overlap':len(set(unique.event_id)&set(training.event_id)),'naive_interval':str(wilson(int((vendor.label==vendor.prediction).sum()),len(vendor))),'unique_interval':str(wilson(int((unique.label==unique.prediction).sum()),len(unique)))}])
save(audit,'vendor_audit.csv')
v=pd.read_csv(ROOT/'vendor/holdout_predictions.csv')
matched=holdout[['event_id','label']].merge(v,on='event_id',validate='one_to_one',suffixes=('_truth','_vendor'))
assert len(matched)==len(holdout) and (matched.label_truth==matched.label_vendor).all()
vendor_result=metrics(matched.label_truth,matched.score,.5)
comparison=pd.concat([results,pd.DataFrame([dict(model='Vendor_C_fixed_0.5',**vendor_result)])],ignore_index=True)
save(comparison,'matched_comparison.csv')
# Same fixed 0.5 threshold is a useful supplementary comparison, not necessarily equal operating cost.
save(pd.DataFrame([dict(model=name,**metrics(holdout.label,p,.5)) for name,p in probabilities.items()]+[dict(model='Vendor_C',**vendor_result)]),'common_threshold_comparison.csv')
# Paired bootstrap difference in accuracy: resample matched event indices, not separate samples.
vindexed=v.set_index('event_id').loc[holdout.event_id]
bcorrect=(probabilities['XGBoost']>=thresholds['XGBoost'])==holdout.label.to_numpy()
vcorrect=vindexed.prediction.to_numpy()==holdout.label.to_numpy()
rng=np.random.default_rng(179);delta=[]
for _ in range(1000):
    idx=rng.integers(0,len(holdout),len(holdout));delta.append(np.mean(bcorrect[idx].astype(int)-vcorrect[idx].astype(int)))
print('XGBoost minus vendor accuracy, paired bootstrap 95% interval:',np.quantile(delta,[.025,.975]))
# Additional supplied black-box stress evidence.
vb=pd.read_csv(ROOT/'vendor/attack_baseline_predictions.csv').set_index('event_id').sort_index()
va=pd.read_csv(ROOT/'vendor/attack_data_predictions.csv').set_index('event_id').sort_index()
assert vb.index.equals(va.index)
print('Vendor baseline/stress recall:',vb.prediction.mean(),va.prediction.mean())

## Governance and example recommendation
Use NIST AI RMF as the required governance lens (the specification allows NIST and/or the EU AI Act). GOVERN: assign owners, approval rules and supplier obligations. MAP: document phishing use case, data provenance and harm. MEASURE: link metrics, uncertainty, explanations and stress evidence. MANAGE: stage deployment, human review, monitoring and rollback.

Example defensible position: do not approve unrestricted production blocking from this evidence. Consider a monitored shadow trial of the selected conventional candidate only after representative recent data, end-to-end load tests, feature availability and operational thresholds are validated. Reject the headline vendor benchmark as sufficient procurement assurance; request independent tests, versioned evidence and supplier commitments. Conditional approval, rejection or a different model can all earn credit if justified.

A high test score does not certify safety or legal compliance. Legal scope is not inferred from this dataset. Learners may discuss EU AI Act applicability, but this reference path does not require a high-risk classification.

In [ ]:
risks=pd.DataFrame([
{'risk':'Old and curated data; contemporary performance unknown','evidence':'2015 donation; curated holdout; no timestamps','severity':'High','mitigation':'Validate recent representative domain-separated data','owner':'Model assurance lead','residual':'High until validation','gate':'Independent acceptance report','rmf':'MAP / MEASURE'},
{'risk':'Feature-space evasion sensitivity','evidence':'robustness.csv','severity':'High','mitigation':'Human review, complementary signals and realistic controlled attack tests','owner':'Detection engineering','residual':'Medium only after controls tested','gate':'Stress-test sign-off','rmf':'MEASURE / MANAGE'},
{'risk':'SOC overload and missed phishing','evidence':'workload.csv; holdout_metrics.csv','severity':'High','mitigation':'Shadow mode, workload monitoring, approved thresholds and rollback','owner':'SOC manager','residual':'Medium pending pilot','gate':'Observed alert capacity and recall targets','rmf':'MANAGE'},
{'risk':'Opaque supplier and biased benchmark','evidence':'vendor_audit.csv; technical appendix','severity':'High','mitigation':'Require independent evidence, dependency inventory, update and incident clauses','owner':'Procurement and security','residual':'High until evidence supplied','gate':'Contract and assurance review','rmf':'GOVERN'},
{'risk':'Subgroup gaps; no demographic evidence','evidence':'subgroups.csv; DATA_CARD.md','severity':'Medium','mitigation':'Collect appropriate deployment cohort evidence and review error disparities','owner':'Governance lead','residual':'Unknown for unobserved groups','gate':'Cohort review','rmf':'MAP / MEASURE'}])
save(risks,'example_residual_risks.csv')
print('Reference analysis complete. Use the three report templates; attach cited evidence, not unexplained tables.')